# Skenario 1 — BM25 Sparse Retrieval (NMT vs LLM)

Notebook ini mengevaluasi model BM25 dan BM25+RM3 menggunakan **dua versi kueri Arab**:
- `query_nmt` — hasil translasi Google Translate (NMT)
- `query_llm` — hasil translasi Gemini AI (LLM)

Evaluasi dilakukan secara:
1. **Overall** — rata-rata seluruh kueri
2. **Per Tipe Kueri** — breakdown berdasarkan `query_type`
3. **Detail Per-Kueri** — Hit/Miss untuk setiap kueri

## 1. Setup & Install Library

In [1]:
import subprocess
import sys
import os

print("Menginstall library...")
!pip install -q python-terrier pyarabic pandas tashaphyne

if os.path.exists('skripsi-clir-code'):
    print("Repository ditemukan. Melakukan pull...")
    !cd skripsi-clir-code && git pull
else:
    print("Cloning repository...")
    !git clone https://github.com/syifaurrr/skripsi-clir-code.git

REPO_PATH = './skripsi-clir-code'
SRC_PATH  = os.path.join(REPO_PATH, 'src')
sys.path.append(SRC_PATH)

import pandas as pd
import pyterrier as pt
from arabic_preprocessing import preprocess_pipeline

if not pt.started():
    pt.init()
print("✅ PyTerrier Started!")

Menginstall library...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.5/251.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.1 MB/s eta 0:00:00
Cloning repository...
Cloning into 'skripsi-clir-code'...
remote: Enumerating objects: 1051, done.
remote: Counting objects: 100% (1051/1051), done.
remote: Compressing objects: 100

/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.12/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)
/tmp/ipykernel_25/1157134227.py:23: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependencies.jar: 100%|██████████| 99.6M/99.6M [00:01<00:00, 63.0MB/s]


Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar: 100%|██████████| 36.6k/36.6k [00:00<00:00, 35.0MB/s]


Done
✅ PyTerrier Started!


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_25/1157134227.py:24: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


## 2. Load & Preprocess Data

In [2]:
DATA_PATH   = './skripsi-clir-code/data'
RAW_DOCS    = os.path.join(DATA_PATH, 'raw/fathul_muin.csv')
QUERY_ARAB  = os.path.join(DATA_PATH, 'queries/queries_arab.csv')
QUERY_INDO  = os.path.join(DATA_PATH, 'queries/queries_indo.csv')
QRELS_PATH  = os.path.join(DATA_PATH, 'queries/qrels.csv')

KOLOM_NMT = 'query_nmt'
KOLOM_LLM = 'query_llm'

# Dokumen
print("1. Memuat & Preprocessing Dokumen...")
df_docs = pd.read_csv(RAW_DOCS)
df_docs.columns = df_docs.columns.str.strip()
df_docs['text_processed'] = df_docs['text'].apply(preprocess_pipeline)
df_docs['docno'] = df_docs['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
print(f"   ✅ {len(df_docs)} dokumen diproses.")

# Kueri Arab
print("\n2. Memuat Kueri Bahasa Arab (NMT & LLM)...")
df_queries_arab = pd.read_csv(QUERY_ARAB)
df_queries_arab['qid'] = df_queries_arab['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

topics_nmt = df_queries_arab[['qid', KOLOM_NMT]].rename(columns={KOLOM_NMT: 'query'}).copy()
topics_llm = df_queries_arab[['qid', KOLOM_LLM]].rename(columns={KOLOM_LLM: 'query'}).copy()
print(f"   Topics NMT: {len(topics_nmt)} kueri")
print(f"   Topics LLM: {len(topics_llm)} kueri")

# Qrels
print("\n3. Memuat Qrels...")
qrels = pd.read_csv(QRELS_PATH)
qrels.columns = qrels.columns.str.strip()

col_map = {}
for col in qrels.columns:
    col_lower = col.lower().strip()
    if col_lower in ('doc_no', 'doc_id', 'docno', 'document_id', 'doc'):
        col_map[col] = 'docno'
    elif col_lower in ('qid', 'query_id', 'query id'):
        col_map[col] = 'qid'
    elif col_lower in ('label', 'relevance', 'rel', 'score'):
        col_map[col] = 'label'
qrels.rename(columns=col_map, inplace=True)

required_cols = {'qid', 'docno', 'label'}
missing = required_cols - set(qrels.columns)
if missing:
    raise ValueError(f"Kolom tidak ditemukan di qrels: {missing}. Kolom tersedia: {qrels.columns.tolist()}")

qrels['qid']   = qrels['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['docno'] = qrels['docno'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
qrels['label'] = qrels['label'].astype(int)
print(f"   ✅ {len(qrels)} qrels dimuat.")

# Metadata tipe kueri
print("\n4. Memuat Metadata Tipe Kueri (Bahasa Indonesia)...")
df_queries_indo = pd.read_csv(QUERY_INDO)
df_queries_indo['qid'] = df_queries_indo['qid'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
print(f"   ✅ {len(df_queries_indo)} kueri indo dimuat.")

print("\n✅ Persiapan Data Selesai!")

1. Memuat & Preprocessing Dokumen...
   ✅ 639 dokumen diproses.

2. Memuat Kueri Bahasa Arab (NMT & LLM)...
   Topics NMT: 153 kueri
   Topics LLM: 153 kueri

3. Memuat Qrels...
   ✅ 153 qrels dimuat.

4. Memuat Metadata Tipe Kueri (Bahasa Indonesia)...
   ✅ 153 kueri indo dimuat.

✅ Persiapan Data Selesai!


## 3. Indexing

In [3]:
index_dir = os.path.join(DATA_PATH, 'indices/skenario_1_sparse')
os.makedirs(index_dir, exist_ok=True)

indexer = pt.IterDictIndexer(
    index_dir,
    meta={'docno': 20, 'text': 4096},
    overwrite=True,
    stemmer=None,
    stopwords=None,
    tokeniser="UTFTokeniser"
)

docs_to_index = df_docs[['docno', 'text_processed']].rename(columns={'text_processed': 'text'}).copy()

print("Membuat Index...")
index_ref = indexer.index(docs_to_index.to_dict(orient='records'))

index = pt.IndexFactory.of(index_ref)
print(f"✅ Index tersimpan di: {index_dir}")
print(f"   Jumlah Dokumen : {index.getCollectionStatistics().getNumberOfDocuments()}")
print(f"   Jumlah Term    : {index.getCollectionStatistics().getNumberOfUniqueTerms()}")
print(f"   Jumlah Token   : {index.getCollectionStatistics().getNumberOfTokens()}")

Membuat Index...
✅ Index tersimpan di: ./skripsi-clir-code/data/indices/skenario_1_sparse
   Jumlah Dokumen : 639
   Jumlah Term    : 8113
   Jumlah Token   : 108920


## 4. Preprocessing Kueri

In [4]:
from IPython.display import display
import numpy as np
from itertools import product

eval_metrics_list = ["recip_rank", "success_10", "success_20", "success_50", "success_100"]

def preprocess_topics(topics, label):
    topics = topics.copy()
    topics["query"] = topics["query"].astype(str).apply(preprocess_pipeline)
    topics["query"] = topics["query"].str.replace(r'[^\w\s]', '', regex=True)
    before = len(topics)
    topics_clean = topics[topics["query"].notna() & (topics["query"].str.strip() != "")].copy()
    dropped = before - len(topics_clean)
    if dropped > 0:
        skipped = topics[~topics["qid"].isin(topics_clean["qid"])]["qid"].tolist()
        print(f"  ⚠️  [{label}] {dropped} kueri kosong setelah preprocessing: qid = {skipped}")
    else:
        print(f"  ✅ [{label}] Semua kueri valid setelah preprocessing.")
    return topics_clean

print("Preprocessing kueri Arab...")
topics_nmt_clean = preprocess_topics(topics_nmt, "Google NMT")
topics_llm_clean = preprocess_topics(topics_llm, "Gemini LLM")

valid_qids = set(topics_nmt_clean["qid"]) | set(topics_llm_clean["qid"])
qrels_filtered = qrels[qrels["qid"].isin(valid_qids)].copy()

def avg_metrics(res_nmt, res_llm):
    """Hitung rata-rata metrik NMT+LLM dari hasil pt.Experiment."""
    return {
        'recip_rank_avg': round((res_nmt['recip_rank'].values[0] + res_llm['recip_rank'].values[0]) / 2, 4),
        'success_10_avg': round((res_nmt['success_10'].values[0]  + res_llm['success_10'].values[0])  / 2, 4),
        'success_20_avg': round((res_nmt['success_20'].values[0]  + res_llm['success_20'].values[0])  / 2, 4),
    }

def run_experiment(retriever, name='model'):
    """Evaluasi retriever pada NMT dan LLM, kembalikan dict metrik rata-rata."""
    res_nmt = pt.Experiment([retriever], topics_nmt_clean,
                            qrels_filtered[qrels_filtered['qid'].isin(topics_nmt_clean['qid'])],
                            eval_metrics=['recip_rank','success_10','success_20'],
                            names=[name], validate='ignore')
    res_llm = pt.Experiment([retriever], topics_llm_clean,
                            qrels_filtered[qrels_filtered['qid'].isin(topics_llm_clean['qid'])],
                            eval_metrics=['recip_rank','success_10','success_20'],
                            names=[name], validate='ignore')
    return avg_metrics(res_nmt, res_llm)

print(f"Topics NMT : {len(topics_nmt_clean)} kueri")
print(f"Topics LLM : {len(topics_llm_clean)} kueri")
print(f"Qrels      : {len(qrels_filtered)} entri")
print("✅ Setup selesai.")

Preprocessing kueri Arab...
  ✅ [Google NMT] Semua kueri valid setelah preprocessing.
  ✅ [Gemini LLM] Semua kueri valid setelah preprocessing.
Topics NMT : 153 kueri
Topics LLM : 153 kueri
Qrels      : 153 entri
✅ Setup selesai.


## 5. Exhaustive Evaluation Parameter BM25

Mengevaluasi semua kombinasi `k1` dan `b` untuk menemukan konfigurasi BM25 terbaik.

In [5]:
k1_list = [0.5, 1.0, 1.2, 1.5, 2.0]
b_list   = [0.0, 0.25, 0.5, 0.75, 1.0]

bm25_results = []
total_bm25 = len(k1_list) * len(b_list)
print(f"Mengevaluasi {total_bm25} kombinasi parameter BM25...\n")

for idx, (k1, b) in enumerate(product(k1_list, b_list), 1):
    retriever = pt.terrier.Retriever(
        index_ref,
        wmodel="BM25",
        controls={"bm25.k_1": k1, "bm25.b": b},
        properties={"termpipelines": ""}
    )
    metrics = run_experiment(retriever)
    bm25_results.append({'k1': k1, 'b': b, **metrics})
    print(f"[{idx:2d}/{total_bm25}] k1={k1}, b={b} "
          f"→ MRR={metrics['recip_rank_avg']:.4f}  "
          f"S@10={metrics['success_10_avg']*100:.1f}%  "
          f"S@20={metrics['success_20_avg']*100:.1f}%")

df_bm25_eval = pd.DataFrame(bm25_results).sort_values('recip_rank_avg', ascending=False).reset_index(drop=True)
print("\n✅ Evaluasi BM25 selesai.")

Mengevaluasi 25 kombinasi parameter BM25...

[ 1/25] k1=0.5, b=0.0 → MRR=0.2558  S@10=41.5%  S@20=49.7%
[ 2/25] k1=0.5, b=0.25 → MRR=0.2532  S@10=41.5%  S@20=50.3%
[ 3/25] k1=0.5, b=0.5 → MRR=0.2557  S@10=41.5%  S@20=50.6%
[ 4/25] k1=0.5, b=0.75 → MRR=0.2557  S@10=41.8%  S@20=50.3%
[ 5/25] k1=0.5, b=1.0 → MRR=0.2565  S@10=41.5%  S@20=49.7%
[ 6/25] k1=1.0, b=0.0 → MRR=0.2533  S@10=39.5%  S@20=49.7%
[ 7/25] k1=1.0, b=0.25 → MRR=0.2534  S@10=40.8%  S@20=49.4%
[ 8/25] k1=1.0, b=0.5 → MRR=0.2514  S@10=40.8%  S@20=50.0%
[ 9/25] k1=1.0, b=0.75 → MRR=0.2508  S@10=40.8%  S@20=50.0%
[10/25] k1=1.0, b=1.0 → MRR=0.2522  S@10=40.8%  S@20=50.0%
[11/25] k1=1.2, b=0.0 → MRR=0.2503  S@10=39.2%  S@20=48.4%
[12/25] k1=1.2, b=0.25 → MRR=0.2506  S@10=40.2%  S@20=48.7%
[13/25] k1=1.2, b=0.5 → MRR=0.2515  S@10=40.2%  S@20=49.0%
[14/25] k1=1.2, b=0.75 → MRR=0.2522  S@10=40.2%  S@20=49.0%
[15/25] k1=1.2, b=1.0 → MRR=0.2535  S@10=40.2%  S@20=49.4%
[16/25] k1=1.5, b=0.0 → MRR=0.2485  S@10=39.9%  S@20=48.4%
[17/2

### Hasil Evaluasi BM25

In [6]:
print("Top 10 Kombinasi Parameter BM25 (diurutkan by MRR rata-rata NMT+LLM)")
display(df_bm25_eval[['k1','b','recip_rank_avg','success_10_avg','success_20_avg']].head(10))

best_bm25 = df_bm25_eval.iloc[0]
best_k1 = float(best_bm25['k1'])
best_b  = float(best_bm25['b'])

print(f"\n🏆 Parameter BM25 Terbaik:")
print(f"   k1  = {best_k1}")
print(f"   b   = {best_b}")
print(f"   MRR = {best_bm25['recip_rank_avg']}")
print(f"   S@10= {round(best_bm25['success_10_avg']*100, 2)}%")
print(f"   S@20= {round(best_bm25['success_20_avg']*100, 2)}%")

Top 10 Kombinasi Parameter BM25 (diurutkan by MRR rata-rata NMT+LLM)


,k1,b,recip_rank_avg,success_10_avg,success_20_avg
0,0.5,1.00,0.2565,0.4150,0.4967
1,0.5,0.00,0.2558,0.4150,0.4967
2,0.5,0.50,0.2557,0.4150,0.5065
3,0.5,0.75,0.2557,0.4183,0.5033
4,1.2,1.00,0.2535,0.4020,0.4935
5,1.0,0.25,0.2534,0.4085,0.4935
6,1.0,0.00,0.2533,0.3954,0.4967
7,0.5,0.25,0.2532,0.4150,0.5033
8,1.0,1.00,0.2522,0.4085,0.5000
9,1.2,0.75,0.2522,0.4020,0.4902



🏆 Parameter BM25 Terbaik:
   k1  = 0.5
   b   = 1.0
   MRR = 0.2565
   S@10= 41.5%
   S@20= 49.67%


## 6. Exhaustive Evaluation Parameter BM25+RM3

Mengevaluasi semua kombinasi `fb_docs`, `fb_terms`, `fb_lambda` dengan parameter BM25 terbaik.

In [7]:
fb_docs_list   = [3, 5, 10]
fb_terms_list  = [5, 10, 20]
fb_lambda_list = [0.3, 0.5, 0.6, 0.8]

bm25_best = pt.terrier.Retriever(
    index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": best_k1, "bm25.b": best_b},
    properties={"termpipelines": ""}
)

rm3_results = []
total_rm3 = len(fb_docs_list) * len(fb_terms_list) * len(fb_lambda_list)
print(f"Mengevaluasi {total_rm3} kombinasi parameter BM25+RM3")
print(f"(BM25 fix: k1={best_k1}, b={best_b})\n")

for idx, (fd, ft, fl) in enumerate(product(fb_docs_list, fb_terms_list, fb_lambda_list), 1):
    retriever = bm25_best >> pt.rewrite.RM3(
        index_ref, fb_docs=fd, fb_terms=ft, fb_lambda=fl
    ) >> bm25_best
    metrics = run_experiment(retriever)
    rm3_results.append({'fb_docs': fd, 'fb_terms': ft, 'fb_lambda': fl, **metrics})
    print(f"[{idx:2d}/{total_rm3}] fb_docs={fd}, fb_terms={ft}, fb_lambda={fl} "
          f"→ MRR={metrics['recip_rank_avg']:.4f}  "
          f"S@10={metrics['success_10_avg']*100:.1f}%  "
          f"S@20={metrics['success_20_avg']*100:.1f}%")

df_rm3_eval = pd.DataFrame(rm3_results).sort_values('recip_rank_avg', ascending=False).reset_index(drop=True)
print("\n✅ Evaluasi BM25+RM3 selesai.")

Mengevaluasi 36 kombinasi parameter BM25+RM3
(BM25 fix: k1=0.5, b=1.0)

[ 1/36] fb_docs=3, fb_terms=5, fb_lambda=0.3 → MRR=0.2145  S@10=31.1%  S@20=40.2%
[ 2/36] fb_docs=3, fb_terms=5, fb_lambda=0.5 → MRR=0.2243  S@10=35.0%  S@20=45.4%
[ 3/36] fb_docs=3, fb_terms=5, fb_lambda=0.6 → MRR=0.2308  S@10=38.2%  S@20=47.7%
[ 4/36] fb_docs=3, fb_terms=5, fb_lambda=0.8 → MRR=0.2495  S@10=44.1%  S@20=51.0%
[ 5/36] fb_docs=3, fb_terms=10, fb_lambda=0.3 → MRR=0.2310  S@10=34.3%  S@20=42.2%
[ 6/36] fb_docs=3, fb_terms=10, fb_lambda=0.5 → MRR=0.2401  S@10=37.9%  S@20=47.4%
[ 7/36] fb_docs=3, fb_terms=10, fb_lambda=0.6 → MRR=0.2462  S@10=40.5%  S@20=51.6%
[ 8/36] fb_docs=3, fb_terms=10, fb_lambda=0.8 → MRR=0.2557  S@10=43.1%  S@20=53.3%
[ 9/36] fb_docs=3, fb_terms=20, fb_lambda=0.3 → MRR=0.2369  S@10=36.6%  S@20=46.4%
[10/36] fb_docs=3, fb_terms=20, fb_lambda=0.5 → MRR=0.2463  S@10=40.8%  S@20=50.3%
[11/36] fb_docs=3, fb_terms=20, fb_lambda=0.6 → MRR=0.2510  S@10=40.8%  S@20=53.9%
[12/36] fb_docs=3, 

### Hasil Evaluasi BM25+RM3

In [8]:
print("Top 10 Kombinasi Parameter BM25+RM3 (diurutkan by MRR rata-rata NMT+LLM)")
display(df_rm3_eval[['fb_docs','fb_terms','fb_lambda','recip_rank_avg','success_10_avg','success_20_avg']].head(10))

best_rm3 = df_rm3_eval.iloc[0]
best_fd = int(best_rm3['fb_docs'])
best_ft = int(best_rm3['fb_terms'])
best_fl = float(best_rm3['fb_lambda'])

print(f"\n🏆 Parameter BM25+RM3 Terbaik:")
print(f"   fb_docs   = {best_fd}")
print(f"   fb_terms  = {best_ft}")
print(f"   fb_lambda = {best_fl}")
print(f"   MRR       = {best_rm3['recip_rank_avg']}")
print(f"   S@10      = {round(best_rm3['success_10_avg']*100, 2)}%")
print(f"   S@20      = {round(best_rm3['success_20_avg']*100, 2)}%")

Top 10 Kombinasi Parameter BM25+RM3 (diurutkan by MRR rata-rata NMT+LLM)


,fb_docs,fb_terms,fb_lambda,recip_rank_avg,success_10_avg,success_20_avg
0,10,20,0.8,0.2594,0.4379,0.5359
1,5,20,0.8,0.2593,0.4379,0.5327
2,5,5,0.8,0.2561,0.4379,0.5131
3,3,10,0.8,0.2557,0.4314,0.5327
4,10,5,0.8,0.2554,0.4314,0.5163
5,3,20,0.8,0.2553,0.4412,0.5327
6,5,10,0.8,0.2544,0.4379,0.5294
7,10,20,0.6,0.2538,0.4020,0.5359
8,10,10,0.8,0.2535,0.4379,0.5359
9,5,20,0.6,0.2533,0.4118,0.5392



🏆 Parameter BM25+RM3 Terbaik:
   fb_docs   = 10
   fb_terms  = 20
   fb_lambda = 0.8
   MRR       = 0.2594
   S@10      = 43.79%
   S@20      = 53.59%


## 7. Evaluasi Final — NMT vs LLM

Evaluasi menggunakan parameter terbaik dari masing-masing model.

In [9]:
bm25 = pt.terrier.Retriever(
    index_ref,
    wmodel="BM25",
    controls={"bm25.k_1": best_k1, "bm25.b": best_b},
    properties={"termpipelines": ""}
)
rm3 = bm25 >> pt.rewrite.RM3(
    index_ref,
    fb_docs=best_fd,
    fb_terms=best_ft,
    fb_lambda=best_fl
) >> bm25

print(f"✅ Model final:")
print(f"   BM25     : k1={best_k1}, b={best_b}")
print(f"   BM25+RM3 : fb_docs={best_fd}, fb_terms={best_ft}, fb_lambda={best_fl}")

def run_perquery_experiment(topics_clean, label_prefix):
    qrels_subset = qrels_filtered[qrels_filtered["qid"].isin(topics_clean["qid"])].copy()
    return pt.Experiment(
        [bm25, rm3],
        topics_clean,
        qrels_subset,
        eval_metrics=eval_metrics_list,
        names=[f"BM25 ({label_prefix})", f"BM25+RM3 ({label_prefix})"],
        validate="ignore",
        perquery=True
    )

print("Menjalankan Evaluasi Per-Kueri (NMT vs LLM) untuk BM25 & BM25+RM3...")
hasil_nmt_perq = run_perquery_experiment(topics_nmt_clean, "Google NMT")
hasil_llm_perq = run_perquery_experiment(topics_llm_clean, "Gemini LLM")

hasil_eval_perq = pd.concat([hasil_nmt_perq, hasil_llm_perq], ignore_index=True)

hasil_pivot = hasil_eval_perq.pivot_table(
    index=["name", "qid"], columns="measure", values="value"
).reset_index()
hasil_pivot.columns.name = None

for m in eval_metrics_list:
    if m not in hasil_pivot.columns:
        hasil_pivot[m] = 0.0

hasil_pivot["qid"] = hasil_pivot["qid"].astype(str)
df_queries_indo["qid"] = df_queries_indo["qid"].astype(str)

hasil_dengan_tipe = pd.merge(
    hasil_pivot,
    df_queries_indo[["qid", "query_type", "query"]],
    on="qid", how="left"
)
print("✅ Evaluasi selesai.")

✅ Model final:
   BM25     : k1=0.5, b=1.0
   BM25+RM3 : fb_docs=10, fb_terms=20, fb_lambda=0.8
Menjalankan Evaluasi Per-Kueri (NMT vs LLM) untuk BM25 & BM25+RM3...
✅ Evaluasi selesai.


### 7.1 Hasil Keseluruhan (Overall)

In [10]:
output_dir = os.path.join(DATA_PATH, 'results', 'skenario1')
os.makedirs(output_dir, exist_ok=True)

overall_metrics = hasil_dengan_tipe.groupby('name')[eval_metrics_list].mean().reset_index()

for k in [10, 20, 50, 100]:
    overall_metrics[f'success_{k} (%)'] = (overall_metrics[f'success_{k}'] * 100).round(2)
overall_metrics['recip_rank'] = overall_metrics['recip_rank'].round(4)

cols_tampil = ['name', 'recip_rank', 'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']
display(overall_metrics[cols_tampil])

overall_metrics.to_csv(os.path.join(output_dir, 'skenario1_overall_nmt_vs_llm.csv'), index=False)
print("📑 Disimpan: skenario1_overall_nmt_vs_llm.csv")

,name,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,BM25 (Gemini LLM),0.3568,56.21,65.36,77.78,90.20
1,BM25 (Google NMT),0.1562,26.80,33.99,53.59,66.01
2,BM25+RM3 (Gemini LLM),0.3602,56.86,66.67,82.35,90.20
3,BM25+RM3 (Google NMT),0.1586,30.72,40.52,52.94,67.97


📑 Disimpan: skenario1_overall_nmt_vs_llm.csv


### 7.2 Hasil Berdasarkan Tipe Kueri

In [11]:
per_type_metrics = (
    hasil_dengan_tipe
    .groupby(['name', 'query_type'])[eval_metrics_list]
    .mean()
    .reset_index()
)

for k in [10, 20, 50, 100]:
    per_type_metrics[f'success_{k} (%)'] = (per_type_metrics[f'success_{k}'] * 100).round(2)
per_type_metrics['recip_rank'] = per_type_metrics['recip_rank'].round(4)

cols_tampil_type = ['name', 'query_type', 'recip_rank',
                    'success_10 (%)', 'success_20 (%)', 'success_50 (%)', 'success_100 (%)']
display(per_type_metrics[cols_tampil_type])

per_type_metrics.to_csv(os.path.join(output_dir, 'skenario1_tipe_kueri_nmt_vs_llm.csv'), index=False)
print("📑 Disimpan: skenario1_tipe_kueri_nmt_vs_llm.csv")

,name,query_type,recip_rank,success_10 (%),success_20 (%),success_50 (%),success_100 (%)
0,BM25 (Gemini LLM),1,0.4512,68.63,76.47,88.24,96.08
1,BM25 (Gemini LLM),2,0.3800,60.78,72.55,76.47,88.24
2,BM25 (Gemini LLM),3,0.2391,39.22,47.06,68.63,86.27
3,BM25 (Google NMT),1,0.2103,35.29,39.22,52.94,62.75
4,BM25 (Google NMT),2,0.1172,23.53,37.25,58.82,64.71
5,BM25 (Google NMT),3,0.1411,21.57,25.49,49.02,70.59
6,BM25+RM3 (Gemini LLM),1,0.4545,70.59,80.39,94.12,96.08
7,BM25+RM3 (Gemini LLM),2,0.3824,62.75,70.59,80.39,90.20
8,BM25+RM3 (Gemini LLM),3,0.2437,37.25,49.02,72.55,84.31
9,BM25+RM3 (Google NMT),1,0.2054,41.18,47.06,54.90,64.71


📑 Disimpan: skenario1_tipe_kueri_nmt_vs_llm.csv


### 7.3 Analisis Detail Per-Kueri

In [12]:
detail_kueri = hasil_dengan_tipe.copy()

detail_kueri['rank_ditemukan'] = detail_kueri['recip_rank'].apply(
    lambda x: int(round(1 / x)) if x > 0 else "Tidak Ketemu"
)

for k in [10, 20, 50, 100]:
    detail_kueri[f'Hit@{k}'] = detail_kueri[f'success_{k}'].apply(
        lambda x: '✅ Hit' if x == 1.0 else '❌ Miss'
    )

detail_kueri['qid_int'] = pd.to_numeric(detail_kueri['qid'], errors='coerce').fillna(0).astype(int)
detail_kueri = detail_kueri.sort_values(by=['name', 'query_type', 'qid_int'])

kolom_tampil = ['name', 'qid', 'query_type', 'query',
                'rank_ditemukan', 'Hit@10', 'Hit@20', 'Hit@50', 'Hit@100']
tabel_detail_final = detail_kueri[kolom_tampil]

display(tabel_detail_final.head(20))

tabel_detail_final.to_csv(
    os.path.join(output_dir, 'skenario1_detail_per_kueri_nmt_vs_llm.csv'), index=False
)
print("📑 Disimpan: skenario1_detail_per_kueri_nmt_vs_llm.csv")

,name,qid,query_type,query,rank_ditemukan,Hit@10,Hit@20,Hit@50,Hit@100
0,BM25 (Gemini LLM),1,1,Kewajiban orang tua untuk memerintahkan sholat...,1,✅ Hit,✅ Hit,✅ Hit,✅ Hit
87,BM25 (Gemini LLM),4,1,Kesunnahan membaca basmalah dalam wudhu,24,❌ Miss,❌ Miss,✅ Hit,✅ Hit
120,BM25 (Gemini LLM),7,1,Alasan-alasan yang membatalkan wudhu,1,✅ Hit,✅ Hit,✅ Hit,✅ Hit
1,BM25 (Gemini LLM),10,1,Syarat sah sholat yang kedua,5,✅ Hit,✅ Hit,✅ Hit,✅ Hit
34,BM25 (Gemini LLM),13,1,Tata cara mensucikan najis (air seni) yang tel...,1,✅ Hit,✅ Hit,✅ Hit,✅ Hit
61,BM25 (Gemini LLM),16,1,Kewajiban mengingatkan seseorang yang pakaiann...,3,✅ Hit,✅ Hit,✅ Hit,✅ Hit
64,BM25 (Gemini LLM),19,1,Kesunahan membaca doa qunut di dalam sholat subuh,31,❌ Miss,❌ Miss,✅ Hit,✅ Hit
68,BM25 (Gemini LLM),22,1,Solusi bagi makmum ketika lupa membaca surah a...,9,✅ Hit,✅ Hit,✅ Hit,✅ Hit
71,BM25 (Gemini LLM),25,1,Hukum melakukan sunnah ab'ad yang terlupakan s...,9,✅ Hit,✅ Hit,✅ Hit,✅ Hit
74,BM25 (Gemini LLM),28,1,Kesunnahan adzan pada selain sholat fardhu,63,❌ Miss,❌ Miss,❌ Miss,✅ Hit


📑 Disimpan: skenario1_detail_per_kueri_nmt_vs_llm.csv


### 7.4 Ringkasan Akhir

In [13]:
print("=" * 70)
print("📋 RINGKASAN AKHIR — Hit Rate Per Model")
print("=" * 70)

for name, group in detail_kueri.groupby('name'):
    total = len(group)
    print(f"\n🔹 {name} ({total} kueri)")
    for k in [10, 20, 50, 100]:
        hits = (group[f'success_{k}'] == 1.0).sum()
        pct  = hits / total * 100
        print(f"   Hit@{k:3d}: {hits:3d}/{total} ({pct:.1f}%)")

print("\n" + "=" * 70)
print("✅ Seluruh laporan berhasil disimpan dalam format CSV!")
print(f"   Lokasi: {output_dir}")
print("   - skenario1_overall_nmt_vs_llm.csv")
print("   - skenario1_tipe_kueri_nmt_vs_llm.csv")
print("   - skenario1_detail_per_kueri_nmt_vs_llm.csv")

📋 RINGKASAN AKHIR — Hit Rate Per Model

🔹 BM25 (Gemini LLM) (153 kueri)
   Hit@ 10:  86/153 (56.2%)
   Hit@ 20: 100/153 (65.4%)
   Hit@ 50: 119/153 (77.8%)
   Hit@100: 138/153 (90.2%)

🔹 BM25 (Google NMT) (153 kueri)
   Hit@ 10:  41/153 (26.8%)
   Hit@ 20:  52/153 (34.0%)
   Hit@ 50:  82/153 (53.6%)
   Hit@100: 101/153 (66.0%)

🔹 BM25+RM3 (Gemini LLM) (153 kueri)
   Hit@ 10:  87/153 (56.9%)
   Hit@ 20: 102/153 (66.7%)
   Hit@ 50: 126/153 (82.4%)
   Hit@100: 138/153 (90.2%)

🔹 BM25+RM3 (Google NMT) (153 kueri)
   Hit@ 10:  47/153 (30.7%)
   Hit@ 20:  62/153 (40.5%)
   Hit@ 50:  81/153 (52.9%)
   Hit@100: 104/153 (68.0%)

✅ Seluruh laporan berhasil disimpan dalam format CSV!
   Lokasi: ./skripsi-clir-code/data/results/skenario1
   - skenario1_overall_nmt_vs_llm.csv
   - skenario1_tipe_kueri_nmt_vs_llm.csv
   - skenario1_detail_per_kueri_nmt_vs_llm.csv
